# core

> schedules, the lists they read, and the notes they leave

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

All of pobblebonk is here, because [honker](https://github.com/russellromney/honker) already has
most of it. honker brings the queue, the durable scheduler table, the streams with a cursor per
reader, the retries and the named locks, all in one SQLite file and all as SQL functions.

What is left is what honker has not got, and it is three things. A pile you add to over a week
that a fire later reads. A gate that stops a fire running when its pile is empty. And a callback,
because honker fires a job and something has to decide what a job means.

There is no daemon. `honker_scheduler_tick(now)` is a SQL function, so `tick` is one call and a
cron entry is the whole scheduler.

In [ ]:
#| export
import json, re, time
from pathlib import Path
from tempfile import mkdtemp

import honker
from fastcore.all import AttrDict, L, first, ifnone, patch
from honker import crontab, every_s

In [ ]:
#| export
FIRES = 'pob.fires'          # the honker queue every schedule fires into
NOTES = 'pob.notes'          # the honker stream a person reads
CATCHUP = ('once', 'all')    # what to do with the fires missed while the machine was off
MAX_TEXT = 20_000            # chars kept of a note body; an answer is a note, not a document

In [ ]:
#| export
_DUR = re.compile(r'([\d.]+)\s*([smhdw])', re.I)
_MULT = dict(s=1, m=60, h=3600, d=86400, w=604800)

def secs(every) -> int:
    "Whole seconds from `'30m'`, `'6h'`, `'1w'`, `'1h30m'`, or a number."
    try: return int(float(every))
    except (TypeError, ValueError): pass
    if not (ms := _DUR.findall(str(every))): raise ValueError(f'not a duration: {every!r}')
    return int(sum(float(n) * _MULT[u.lower()] for n, u in ms))

def clip(s, mx:int=MAX_TEXT) -> str:
    'Text short enough to store, saying what it dropped rather than losing it quietly.'
    s = '' if s is None else str(s)
    return s if len(s) <= mx else s[:mx] + f'\n... [{len(s)-mx} more chars]'

In [ ]:
test_eq(secs('30m'), 1800)
test_eq(secs('1h30m'), 5400)
test_eq(secs('2 days'), 172800)
test_eq(secs(90), 90)
test_fail(lambda: secs('soon'), contains='not a duration')
test_eq(clip('abcdef', 3), 'abc\n... [3 more chars]')

## Pob

In [ ]:
#| export
class Pob:
    "A honker database, the one table honker has not got, and the callbacks a fire runs."
    def __init__(self,
                 path=None,   # the file; None makes a temporary one
                 db=None):    # or a honker database something else already opened
        # honker watches `PRAGMA data_version` on a file, so `:memory:` is not an option
        if db is None and path is None: path = Path(mkdtemp())/'pob.db'
        self.db = db if db is not None else honker.open(str(path))
        self.q, self.stream = self.db.queue(FIRES), self.db.stream(NOTES)
        self.sched, self.runners = honker.Scheduler(self.db), {}
        with self.db.transaction() as tx:
            tx.execute('CREATE TABLE IF NOT EXISTS pob_items (id INTEGER PRIMARY KEY, topic TEXT, '
                       'text TEXT, key TEXT, at REAL, used_at REAL, used_by TEXT)')
            tx.execute('CREATE INDEX IF NOT EXISTS pob_items_open ON pob_items(topic, at) '
                       'WHERE used_at IS NULL')
            # the same photo sent twice is one item, until something uses it
            tx.execute('CREATE UNIQUE INDEX IF NOT EXISTS pob_items_key ON pob_items(topic, key) '
                       'WHERE key IS NOT NULL AND used_at IS NULL')

    def __repr__(self):
        return f'Pob({len(self.all())} schedules, {self.due()} due, {len(self.notes())} notes)'

## Schedules

`add` writes a row in honker's own scheduler table, so it survives the process and can be paused
from anywhere. Everything pobblebonk needs that honker does not know about rides in the payload.

In [ ]:
#| export
@patch
def add(self:Pob,
        name:str,             # what you call it; unique, and what a note is titled with
        cron:str=None,        # a five-field cron expression
        every=None,           # or a duration: `'30m'`, `'1w'`, or seconds
        needs:str=None,       # a list topic this cannot run without
        catchup:str='once',   # `once` runs the latest missed fire, `all` runs every one
        **payload             # anything the callback wants
) -> AttrDict:
    'Write down a standing instruction.'
    if bool(cron) == bool(every): raise ValueError('a schedule needs one of `cron` or `every`')
    if catchup not in CATCHUP: raise ValueError(f'catchup must be one of {CATCHUP}: {catchup!r}')
    self.sched.add(name=str(name), queue=FIRES, payload=dict(
        schedule=str(name), needs=needs, catchup=catchup, **payload),
        schedule=crontab(cron) if cron else every_s(secs(every)))
    return self.get(name)

@patch
def on(self:Pob, name:str):
    'Decorator naming the callback a fire of `name` runs.'
    def _f(fn):
        self.runners[str(name)] = fn
        return fn
    return _f

@patch
def all(self:Pob) -> L:
    'Every schedule, with its payload parsed.'
    return L(self.sched.list()).map(lambda r: AttrDict(r, payload=json.loads(r['payload'] or 'null')))

@patch
def get(self:Pob, name:str) -> AttrDict: return first(r for r in self.all() if r['name'] == str(name))
@patch
def pause(self:Pob, name:str) -> bool: return self.sched.pause(str(name))
@patch
def resume(self:Pob, name:str) -> bool: return self.sched.resume(str(name))
@patch
def drop(self:Pob, name:str) -> bool:
    self.runners.pop(str(name), None)
    return self.sched.remove(str(name))
@patch
def due(self:Pob) -> int:
    'Fires queued and not yet run.'
    return self.db.query("SELECT count(*) n FROM _honker_live WHERE queue=? AND state='ready'",
                         [FIRES])[0]['n']

## The lists a fire reads

Reading a list never empties it. Only the work that finished does that, so a cart run that failed
loses no milk.

In [ ]:
#| export
@patch
def push(self:Pob,
         topic:str,      # which list
         text:str,       # the line the callback reads
         key:str=None,   # send the same thing twice and it stays one item
         at:float=None
) -> AttrDict:
    'Add one item, or give back the open one that `key` already names.'
    at = ifnone(at, time.time())
    with self.db.transaction() as tx:
        if key is not None and (was := tx.query(
                'SELECT * FROM pob_items WHERE topic=? AND key=? AND used_at IS NULL',
                [str(topic), key])): return AttrDict(was[0])
        got = tx.query('INSERT INTO pob_items (topic, text, key, at) VALUES (?,?,?,?) RETURNING *',
                       [str(topic), clip(text), key, at])
    return AttrDict(got[0])

@patch
def items(self:Pob, topic:str, limit:int=200) -> L:
    'The open items on `topic`, oldest first.'
    return L(self.db.query('SELECT * FROM pob_items WHERE topic=? AND used_at IS NULL '
                           'ORDER BY at LIMIT ?', [str(topic), limit])).map(AttrDict)

@patch
def used(self:Pob, ids, by:str='', at:float=None) -> int:
    'Mark items done, once the work that consumed them succeeded.'
    if not (ids := L(ids)): return 0
    qs = ','.join('?'*len(ids))
    with self.db.transaction() as tx:
        return len(tx.query(f'UPDATE pob_items SET used_at=?, used_by=? WHERE id IN ({qs}) '
                            'AND used_at IS NULL RETURNING id',
                            [ifnone(at, time.time()), str(by), *ids]))

## The notes a person reads

Nothing is sent anywhere. An agent cannot start a turn of its own, so a fire leaves a note and
each place you work drains it on its next turn. honker's stream keeps a cursor per reader, so a
phone and a terminal each see everything once.

In [ ]:
#| export
@patch
def note(self:Pob, title:str, body:str='', **meta):
    'Leave one note.'
    self.stream.publish(dict(title=str(title), body=clip(body), **meta))

def _note(r) -> AttrDict: return AttrDict(json.loads(r['payload']), offset=r['offset'])

@patch
def notes(self:Pob, limit:int=50) -> L:
    'Every note, newest first, whatever any reader has seen.'
    return L(self.db.query('SELECT offset, payload FROM _honker_stream WHERE topic=? '
                           'ORDER BY offset DESC LIMIT ?', [NOTES, limit])).map(_note)

@patch
def drain(self:Pob, reader:str, limit:int=50) -> L:
    'Every note this reader has not had. What a turn calls before it answers you.'
    got = L(self.db.query('SELECT offset, payload FROM _honker_stream WHERE topic=? AND offset>? '
                          'ORDER BY offset LIMIT ?',
                          [NOTES, self.stream.get_offset(str(reader)), limit]))
    if got: self.stream.save_offset(str(reader), got[-1]['offset'])
    return got.map(_note)

## The tick

One SQL call fires what is due and advances every `next_fire_at`. honker enqueues a job for every
missed boundary, so `once` keeps the latest of them and cancels the rest.

In [ ]:
#| export
@patch
def tick(self:Pob,
         at:float=None,        # when this tick is happening; now when unset
         worker:str='tick',    # the lease holder recorded on each job
         run:bool=True         # False queues the fires and runs none of them
) -> AttrDict:
    'Fire what is due, then run it. This is what a cron entry calls.'
    at = int(ifnone(at, time.time()))
    with self.db.transaction() as tx:
        fires = L(json.loads(tx.query('SELECT honker_scheduler_tick(?) AS f', [at])[0]['f']))
    once = {s['name'] for s in self.all() if (s.payload or {}).get('catchup') != 'all'}
    dropped = set()
    for name in {f['name'] for f in fires} & once:
        for f in fires.filter(lambda f: f['name'] == name)[:-1]:
            if self.q.cancel(f['job_id']): dropped.add(f['job_id'])
    kept = fires.filter(lambda f: f['job_id'] not in dropped)
    when = {f['job_id']: f['fire_at'] for f in kept}
    return AttrDict(fired=kept, cancelled=len(dropped),
                    ran=self.work(worker, at, when=when) if run else L())

@patch
def work(self:Pob, worker:str='tick', at:float=None, limit:int=100, when:dict=None) -> L:
    'Claim and run every fire that is due.'
    at, when, out = int(ifnone(at, time.time())), when or {}, L()
    while len(out) < limit:
        if (job := self.q.claim_one(worker)) is None: break
        out.append(self._run(job, at, when.get(job.id)))
    return out

@patch
def _run(self:Pob, job, at:int, fire_at:int=None) -> AttrDict:
    'One fire: gather its list, call the callback, keep the answer, leave a note.'
    p = AttrDict(job.payload or {})
    name = p.get('schedule') or ''
    if (fn := self.runners.get(name)) is None:
        job.fail(err := f'no callback registered for {name!r}')
        return AttrDict(name=name, status='error', error=err)
    got = self.items(p.needs) if p.get('needs') else L()
    if p.get('needs') and not got:
        job.ack()
        return AttrDict(name=name, status='skipped', why=f'the {p.needs} list is empty')
    # `list`, not `items`: `AttrDict.items` is the dict method, and a callback would get that
    fire = AttrDict(p, at=at, fire_at=ifnone(fire_at, job.created_at), list=got, job=job.id)
    try: res = fn(fire)
    except Exception as e:
        job.fail(err := f'{type(e).__name__}: {e}')
        return AttrDict(name=name, status='error', error=err)
    self.q.save_result(job.id, res)
    if got: self.used(got.attrgot('id'), by=name)
    self.note(name, res if isinstance(res, str) else json.dumps(res, default=str),
              ref=job.id, used=len(got))
    job.ack()
    # `used`, not `items`: `AttrDict.items` is the dict method, so that key can never be read back
    return AttrDict(name=name, status='ok', result=res, used=len(got))

## A callback, at a cadence

The headline. A real callback, a real one-second schedule, three real ticks. `fire_at` is honker's
own boundary, so the gaps say whether the cadence held.

In [ ]:
import time

pob = Pob()
beats = []

@pob.on('heartbeat')
def beat(fire):
    beats.append(fire.fire_at)
    return f'beat {len(beats)}'

pob.add('heartbeat', every='1s')
for _ in range(3):
    time.sleep(1.05)
    out = pob.tick()
    test_eq([r.status for r in out.ran], ['ok'])

test_eq(len(beats), 3)
test_eq(beats, sorted(set(beats)))                      # every fire is a distinct boundary
# a tick that arrives a little late reports the newest boundary and cancels the one it skipped,
# so a gap can be more than a second but never less: the cadence is a floor, not a promise
test(min(b-a for a, b in zip(beats, beats[1:])), 1, lambda a, b: a >= b)

The answers reached the notes, and every reader gets them once:

In [ ]:
test_eq(pob.drain('leela').attrgot('body'), ['beat 1', 'beat 2', 'beat 3'])
test_eq(pob.drain('leela'), [])                                    # leela is up to date
test_eq(len(pob.drain('phone')), 3)                                # the phone has seen nothing yet

## Catching up

A laptop shut for ten seconds is ten missed one-second fires. `once` runs the latest and cancels
the rest, `all` runs every one.

In [ ]:
for mode, want in (('once', 1), ('all', 10)):
    p = Pob()
    seen = []
    p.on('daily')(lambda fire: seen.append(fire.fire_at) or 'ok')
    p.add('daily', every='1s', catchup=mode)
    out = p.tick(at=int(time.time()) + 10)
    test_eq(len(out.ran), want)
    test_eq(len(seen), want)
    test_eq(out.cancelled, 10 - want)

# `all` ran every boundary, and they are exactly one second apart
test_eq([b-a for a, b in zip(seen, seen[1:])], [1]*9)

## A list a fire reads

The shopping flow. Push whenever you think of something, and the fire reads the pile.

In [ ]:
pob2 = Pob()

@pob2.on('cart')
def cart(fire):
    return 'added: ' + ', '.join(i.text for i in fire.list)

pob2.add('cart', every='1s', needs='shopping')
pob2.push('shopping', 'yoghurt, the greek one', key='img_2201.heic')
pob2.push('shopping', 'yoghurt, the greek one', key='img_2201.heic')   # the same photo twice
pob2.push('shopping', 'oat milk')
test_eq(pob2.items('shopping').attrgot('text'), ['yoghurt, the greek one', 'oat milk'])

`fire.list` is the pile, not `dict.items`. That collision cost a real bug, so it is asserted:

In [ ]:
time.sleep(1.05)
got = pob2.tick().ran[0]
test_eq(got.status, 'ok')
test_eq(got.result, 'added: yoghurt, the greek one, oat milk')
test_eq(got.used, 2)
test_eq(pob2.items('shopping'), [])              # used, but only because the callback returned

A fire whose list emptied in the meantime is skipped, not failed. There is nothing wrong and
nothing to say:

In [ ]:
time.sleep(1.05)
got = pob2.tick().ran[0]
test_eq(got.status, 'skipped')
test_eq(got.why, 'the shopping list is empty')
test_eq(len(pob2.notes()), 1)                    # and no second note

## A callback that raises

honker keeps the job for a retry, the list keeps its items, and nothing is reported as an answer:

In [ ]:
pob3 = Pob()
pob3.on('cart')(lambda fire: 1/0)
pob3.add('cart', every='1s', needs='shopping')
pob3.push('shopping', 'milk')
time.sleep(1.05)
got = pob3.tick().ran[0]
test_eq(got.status, 'error')
test_eq(got.error, 'ZeroDivisionError: division by zero')
test_eq(pob3.items('shopping').attrgot('text'), ['milk'])   # still there for the retry
test_eq(pob3.notes(), [])

A schedule with no callback registered fails loudly rather than vanishing:

In [ ]:
pob4 = Pob()
pob4.add('nobody', every='1s')
time.sleep(1.05)
test_eq(pob4.tick().ran[0].error, "no callback registered for 'nobody'")

## Pausing, dropping, and what is refused

In [ ]:
pob5 = Pob()
pob5.on('cart')(lambda fire: 'ok')
pob5.add('cart', every='1s')
test_eq(pob5.pause('cart'), True)
time.sleep(1.05)
test_eq(pob5.tick().ran, [])
test_eq(pob5.resume('cart'), True)
test_eq(pob5.drop('cart'), True)
test_eq(pob5.all(), [])

In [ ]:
test_fail(lambda: pob5.add('bad', every='1h', cron='0 17 * * *'), contains='one of')
test_fail(lambda: pob5.add('bad'), contains='one of')
test_fail(lambda: pob5.add('bad', every='1h', catchup='maybe'), contains='catchup must be')
test_fail(lambda: pob5.add('bad', cron='every friday'), contains='cron')

## Sharing the file

Every method takes a honker database rather than opening one, so pobblebonk can live in a file
something else already has open. The fire and the write that caused it commit together.

In [ ]:
import honker

db = honker.open(str(Path(mkdtemp())/'app.db'))
with db.transaction() as tx:
    tx.execute('CREATE TABLE docs (id TEXT PRIMARY KEY, text TEXT)')

pob6 = Pob(db=db)
pob6.on('index')(lambda fire: 'indexed')
pob6.add('index', every='1s')
test_eq(pob6.db is db, True)
test_eq(len(pob6.all()), 1)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()